# AeroLab: portable training sessions

Build on your Linux workstation; train the same headless C# simulator here. CPU is the default. This notebook does not qualify a model and cannot guarantee Colab runtime availability. Only open checkpoint bundles you trust.

Use Runtime → Change runtime type → CPU. Keep the training bundle in Drive. Runtime files are temporary; verified snapshots are copied to Drive. A browser disconnect or laptop shutdown does not guarantee the Colab job will continue.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/AeroLab')
DRIVE.mkdir(parents=True, exist_ok=True)
BUNDLE = DRIVE / 'AeroLab-training.zip'
WORK = Path('/content/AeroLab')
assert BUNDLE.is_file(), f'Upload the verified training bundle to {BUNDLE}' 

## Verify and extract
Paste the SHA-256 from the accompanying `.sha256.json`. The extraction refuses path traversal, symlinks, duplicate files and overwrite. All files are checked before extraction. Use a fresh WORK directory when restoring a different bundle.

In [ ]:
import hashlib, json, zipfile, stat
EXPECTED_SHA256 = 'PASTE_SHA256_HERE'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_SHA256
with zipfile.ZipFile(BUNDLE) as z:
    manifest = json.loads(z.read('bundle.json'))
    names = z.namelist()
    assert len(names) == len(set(names)) and set(names) == {'bundle.json', *manifest['files']}
    for item in z.infolist():
        p = Path(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in item.filename
        assert not stat.S_ISLNK(item.external_attr >> 16)
    for name, meta in manifest['files'].items():
        assert hashlib.sha256(z.read(name)).hexdigest() == meta['sha256']
        assert not (WORK / name).exists(), f'Existing file: {name}'
    for name, meta in manifest['files'].items():
        target = WORK / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(name))
        target.chmod(meta['mode'] & 0o777)
print('Verified:', manifest['kind'], 'run:', manifest.get('run'))

In [ ]:
import subprocess, os
def run(args):
    subprocess.run(args, cwd=WORK, check=True)
# Do not replace Colab's system Python.
run(['bash', 'tools/setup-delivery-ai.sh', '--runtime-only'])
run(['.venv/bin/python', '-c', 'import torch, mlagents.trainers; print(torch.__version__, mlagents.trainers.__version__)'])

## Compatibility gate
Run a two-episode reference check, then a short trainer connection/checkpoint run under a unique tag. If shared libraries or the communicator fail, retain the log and continue on the workstation. Do not rebuild the Unity player in Colab or silently change SDK versions.

In [ ]:
run(['.venv/bin/python', 'tools/evaluate-delivery.py', '--mode', 'reference', '--split', 'diagnostic', '--seed-start', '7000', '--stage', '1', '--episodes', '2', '--name', 'colab-compat-reference'])
run(['.venv/bin/python', 'tools/train-delivery.py', 'flight', '--seed', '11', '--stage', '0', '--steps', '128', '--small-buffer', '--checkpoint-interval', '32', '--tag', 'colab-compat', '--backup-dir', str(DRIVE / 'snapshots')])

## Start or resume a session
Set the exact original arguments for the experiment, including total target steps. Set RESUME only if the bundle contains that run and verified snapshots. Resume continues saved optimizer/model progress; in-flight episodes and unsaved rollout data restart. Keep one writer per experiment and transfer the newest checkpoint before switching machines.

In [ ]:
RESUME = False  # True when continuing this exact run after a previous session.
RUN_TAG = 'colab-dispatch-v1'  # Choose a new tag for a new experiment.
RUN_ID = RUN_TAG + '-dispatch-s11-stage3-noisy'
ARGS = ['dispatch', '--seed', '11', '--stage', '3', '--steps', '32768', '--time-scale', '1000', '--batch-size', '32', '--buffer-size', '128', '--checkpoint-interval', '128', '--summary-freq', '128', '--noisy', '--flight-model', 'Models/continuity-ppo-v2-flight-s11', '--tag', RUN_TAG]
if RESUME and not (WORK / 'results' / RUN_ID / 'run.json').exists():
    # Recover even if the last Colab runtime disappeared before a zip was exported.
    run(['.venv/bin/python', 'tools/restore-delivery-backup.py', '--backup-dir', str(DRIVE / 'snapshots'), '--run', RUN_ID])
command = ['.venv/bin/python', 'tools/train-delivery.py', *ARGS, '--session-minutes', '120', '--backup-dir', str(DRIVE / 'snapshots')]
if RESUME:
    command.append('--resume')
run(command)


## Export for the workstation
Create a checkpoint bundle for later training and copy the candidate ONNX separately for Unity inference. Returning an ONNX does not transfer optimizer state. After importing a candidate on the workstation, rebuild and evaluate in native Unity with no teacher actions.

In [ ]:
OUTPUT = DRIVE / (RUN_ID + '-checkpoint.zip')
run(['.venv/bin/python', 'tools/delivery-transfer.py', 'pack', '--kind', 'checkpoint', '--run', RUN_ID, '--output', str(OUTPUT)])
import shutil
shutil.copy2(WORK / 'results' / RUN_ID / 'DeliveryDispatch.onnx', DRIVE / (RUN_ID + '.onnx'))
print(OUTPUT)

On the workstation, stop the old run, then restore with `python tools/delivery-transfer.py unpack <checkpoint.zip> --destination . --restore-run`. The source, player and original arguments must match. Never run the same experiment on both machines at once.